[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Automatic Differentiation


## What you will be able to do

Say what PyTorch and TensorFlow record while a value is computed, get a gradient from that record
in each framework, and avoid the mistakes that add a new gradient to an old one, change a weight
while it is recorded, or use a record again after it has been freed.

## The idea

### The problem

The **Why PyTorch and TensorFlow** notebook let both frameworks work out gradients without looking at
how they do it, which is enough while a loop uses the record the way the framework expects. A
training loop computes a loss and its gradients thousands of times, and each time the framework
records the computation and then lets the record go. A loop that uses the record any other way
either stops with an error or carries on with the wrong numbers.

Some of those mistakes look like ordinary code. Computing a loss once and calling `backward()` on it
in every step of a loop fails on the second step. Changing a weight in place fails too. Calling
`backward()` again without clearing the last gradient raises nothing, because the new gradient is
added to the old one, and the step that uses it is too large.

### A formal definition

> **Reverse-mode automatic differentiation** records each operation as a value is computed,
> building a graph from the inputs to the result. It then walks that graph from the result back to the
> inputs, multiplying each operation's own derivative by the chain rule, and so gives the gradient of
> the result with respect to every input in one pass. In PyTorch the graph is attached to the
> tensors themselves. In TensorFlow it is written onto a **gradient tape** while the tape is open.

### Why it works that way

In PyTorch, a tensor made with `requires_grad=True` is a **leaf**. Every tensor computed from a leaf
carries a `grad_fn`, the operation that made it, linked to the operations before that. `backward()`
walks those links and stores a gradient in the `.grad` of each leaf. To walk back through a
multiplication or a square, the framework needs the values that went into it, so it saves them,
and frees them once the walk is done, so that memory does not grow with every step. A second
walk of the same graph finds them gone.

`backward()` adds its result to whatever `.grad` already holds. That lets gradients from several
pieces of data be summed, and means a loop must clear them before each step. Changing a leaf in
place while PyTorch is recording raises an error, even when no graph uses the leaf yet. The change
would be recorded as an operation, and a leaf, where gradients are collected, cannot also be the
result of one. `torch.no_grad()` turns recording off for the update.

TensorFlow records what happens inside a `with tf.GradientTape()` block to every trainable `tf.Variable`
used there, and to a constant only if `tape.watch` is called on it. `tape.gradient` returns the gradients
rather than storing them, so nothing adds up. A tape gives one set of gradients and is then done,
unless it is made with `persistent=True`.

### Where this shows up

The **Gradient Descent by Hand** notebook builds a loop from these pieces, the **Losses and
Optimizers** notebook hands the update and the clearing to an optimizer, and the **Training Loops**
notebook uses `torch.no_grad()` for every validation pass.

### What this notebook covers

- the record behind a loss, read back from its `grad_fn`
- `backward()` and `.grad`, checked against derivatives worked out by hand
- gradients that add up, used on purpose
- `torch.no_grad()` and `detach()`
- TensorFlow's tape, a watched constant and a persistent tape
- a gradient through an `if`

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x
print(y)
y.backward()
print(x.grad)
```

```
tensor(15., grad_fn=<AddBackward0>)
tensor(8.)
```

```python
import tensorflow as tf

x = tf.Variable(3.0)
with tf.GradientTape() as tape:
    y = x ** 2 + 2 * x
print(tape.gradient(y, x))
```

```
tf.Tensor(8.0, shape=(), dtype=float32)
```

`y` remembers that an addition made it, `AddBackward0`, and the operations before that. The
derivative of x² + 2x is 2x + 2, which is 8 at x = 3, and both frameworks found it from the record.

## Setup

Three libraries, a way to quiet TensorFlow while it loads, and the stations.

- `numpy` builds the stations
- `torch` is PyTorch
- `tensorflow` is TensorFlow
- `contextlib` and `os` hide the messages TensorFlow prints while it loads

TensorFlow's start-up messages carry the time and a process number, so Setup hides them with
`TF_CPP_MIN_LOG_LEVEL` and `c_messages_hidden`, as in the **Why PyTorch and TensorFlow** notebook.

`station_network_arrays` makes the same made-up weather stations as that notebook, drawn from a
fixed seed so they are the same on every run, and the cell splits them into 800 training, 200
validation and 200 test stations. This notebook uses `scaled`, the stations' features standardized
with the training stations' means, and `jan_mean`, each station's January mean in degrees Celsius,
to make the same straight line from latitude that the **Why PyTorch and TensorFlow** notebook fitted.

In [1]:
import contextlib
import os

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | 1200 stations


## Worked examples

### The record behind a loss

The line from the **Why PyTorch and TensorFlow** notebook, at a slope and an intercept of 0, and the
mean squared error it makes on the training stations:

In [2]:
latitude = torch.from_numpy(scaled[train_rows, 0])
january = torch.from_numpy(jan_mean[train_rows])

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
predicted = w * latitude + b
loss = ((predicted - january) ** 2).mean()

print(w.is_leaf, predicted.is_leaf, loss.is_leaf)
print(predicted.requires_grad, predicted.grad_fn.name(), loss.grad_fn.name())

True False False
True AddBackward0 MeanBackward0


`w` is a leaf, made by hand, and `predicted` and `loss` are not, since each was computed.
`predicted` requires a gradient because a leaf that does went into it. Its `grad_fn` is
`AddBackward0`, the addition of `b` that made it, and the loss's is `MeanBackward0`.

In [3]:
def steps_back(tensor):
    """The names of the operations behind a tensor, following the first input of each."""
    names, step = [], tensor.grad_fn
    while step is not None:
        names.append(step.name())
        inputs = [fn for fn, _ in step.next_functions if fn is not None]
        step = inputs[0] if inputs else None
    return names


print(steps_back(loss))

multiply = predicted.grad_fn.next_functions[0][0]
print(multiply.name(), [fn.name() if fn is not None else None for fn, _ in multiply.next_functions])

['MeanBackward0', 'PowBackward0', 'SubBackward0', 'AddBackward0', 'MulBackward0', 'torch::autograd::AccumulateGrad']
MulBackward0 ['torch::autograd::AccumulateGrad', None]


Read from the loss back to `w`, the record is a mean, of a square, of a subtraction, of the addition
of `b`, of the multiplication by `w`. The last step, `AccumulateGrad`, is where a gradient arriving at
a leaf is added to its `.grad`. `steps_back` follows only the first input of each operation, and the
graph also branches to `b`. The data needs no gradient, so the graph has no step for it, and
`next_functions` holds `None` in its place, as the second line shows for the multiplication, whose
inputs are `w`, through `AccumulateGrad`, and the latitudes.

### backward() fills .grad

In [4]:
loss.backward()
print(w.grad, b.grad)

tensor(3.9696) tensor(10.2153)


The walk back from the loss filled `w.grad` and `b.grad` with 3.9696 and 10.2153, the gradients the
**Why PyTorch and TensorFlow** notebook checked against the formula.

### Checked against derivatives worked out by hand

The derivative of x³ − 2x is 3x² − 2, and the cell below checks both frameworks against it at five
values of x at once.

In [5]:
xs = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0], requires_grad=True)
(xs ** 3 - 2 * xs).sum().backward()
print(xs.grad)
print(3 * xs.detach() ** 2 - 2)

xs_tf = tf.Variable([-2.0, -1.0, 0.0, 1.0, 2.0])
with tf.GradientTape() as tape:
    total = tf.reduce_sum(xs_tf ** 3 - 2 * xs_tf)
print(tape.gradient(total, xs_tf))

tensor([10.,  1., -2.,  1., 10.])
tensor([10.,  1., -2.,  1., 10.])
tf.Tensor([10.  1. -2.  1. 10.], shape=(5,), dtype=float32)


All three rows agree, at 10, 1, −2, 1 and 10. Summing the five results gives one number to call
`backward()` on, and since each term depends on one value of x alone, the gradient of the sum is
each term's own derivative.

### Gradients that add up

`backward()` adds to `.grad`. The cell computes the gradient of the summed squared error over all 800
training stations, then clears it by setting `.grad` to `None`, then computes it from the first 400
and the other 400 in two calls.

In [6]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
((w * latitude + b - january) ** 2).sum().backward()
print("all 800 at once:", round(w.grad.item(), 2), round(b.grad.item(), 2))

w.grad, b.grad = None, None
((w * latitude[:400] + b - january[:400]) ** 2).sum().backward()
((w * latitude[400:] + b - january[400:]) ** 2).sum().backward()
print("400, then 400:  ", round(w.grad.item(), 2), round(b.grad.item(), 2))

all 800 at once: 3175.67 8172.2
400, then 400:   3175.67 8172.2


The two halves add up to the gradient of the whole, to the two decimals printed. Accumulation
exists so that a batch too large to compute at once can be split, with `backward()` on each part
before one update. It is also why a loop clears `.grad` before each step, by setting it to
`None` or with `zero_()`, as the last of the Common errors shows.

### Leaving the record

`torch.no_grad()` turns recording off inside its block, and `detach()` returns a tensor's values with
no record behind them.

In [7]:
with torch.no_grad():
    test_predicted = w * torch.from_numpy(scaled[test_rows, 0]) + b
print(test_predicted.requires_grad, test_predicted.grad_fn)

doubled = (w * 2).detach()
print(doubled.requires_grad, doubled.grad_fn)

False None
False None


Neither tensor requires a gradient or has a `grad_fn`, so nothing computed from them is recorded.
`torch.no_grad()` is for work that is not part of the model, such as an update or a pass over the
test stations, and `detach()` is for keeping a value, to print or to turn into a NumPy array,
without the record it came with.

### TensorFlow's tape

The same line and loss in TensorFlow. The tape watches the two variables without being told.

In [8]:
w_tf = tf.Variable(0.0)
b_tf = tf.Variable(0.0)
latitude_tf = tf.constant(scaled[train_rows, 0])
january_tf = tf.constant(jan_mean[train_rows])

with tf.GradientTape() as tape:
    loss_tf = tf.reduce_mean((w_tf * latitude_tf + b_tf - january_tf) ** 2)
print(len(tape.watched_variables()), "variables watched")

dw, db = tape.gradient(loss_tf, [w_tf, b_tf])
print(round(float(dw), 4), round(float(db), 4))

2 variables watched
3.9696 10.2153


The tape found the same gradients as PyTorch, 3.9696 and 10.2153. `tape.gradient` takes the result
and a list of variables, and returns the gradient of the result with respect to each, in the same
order, without storing them anywhere.

In [9]:
side = tf.constant(1.0)
with tf.GradientTape() as tape:
    tape.watch(side)
    area = 3 * side ** 2
print(tape.gradient(area, side))

v = tf.Variable(2.0)
with tf.GradientTape(persistent=True) as tape:
    square = v ** 2
    cube = v ** 3
print(float(tape.gradient(square, v)), float(tape.gradient(cube, v)))
del tape

tf.Tensor(6.0, shape=(), dtype=float32)
4.0 12.0


A constant is watched only when `tape.watch` is called on it inside the block, and then the
derivative of 3s² at s = 1 is 6. A persistent tape gives two sets of gradients from one recording,
2v = 4 and 3v² = 12 at v = 2, and `del tape` frees what it holds once you are done.

### A gradient through an if

The record holds the operations that ran, so Python's `if`, loops and function calls need nothing
special. This function squares a positive number and negates any other:

In [10]:
def bend(t):
    if t > 0:
        return t ** 2
    return -t


for value in (2.0, -3.0):
    t = torch.tensor(value, requires_grad=True)
    bend(t).backward()

    t_tf = tf.Variable(value)
    with tf.GradientTape() as tape:
        result = bend(t_tf)
    print(value, "PyTorch:", t.grad.item(), "| TensorFlow:", float(tape.gradient(result, t_tf)))

2.0 PyTorch: 4.0 | TensorFlow: 4.0
-3.0 PyTorch: -1.0 | TensorFlow: -1.0


At 2 the square ran, whose derivative 2t is 4, and at −3 the negation ran, whose derivative is −1.
Each run records only the branch it took.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/04-automatic-differentiation-solutions.ipynb).

**1.** Find the derivative of y = 5x² − 3x at x = 2 with PyTorch and with TensorFlow, and check it against 10x − 3.

In [11]:
# your code here

**2.** Make `x = torch.linspace(-2, 2, 5, requires_grad=True)`, find the gradient of `(x ** 3).sum()`, and check it against 3x² with `torch.allclose`.

In [12]:
# your code here

**3.** Print the operations behind `torch.tensor(2.0, requires_grad=True).exp() * 3` with `steps_back`, defined again in your cell, and find its gradient, which should equal 3e².

In [13]:
# your code here

**4.** With one persistent tape, find the gradients of `v ** 2` and of `tf.sin(v)` at v = 0.5, and check them against 2v and cos v.

In [14]:
# your code here

**5.** Compute the mean squared error of the line with a slope of −2 and an intercept of −5 on the test stations inside `torch.no_grad()`, and print the error and its `requires_grad`.

In [15]:
# your code here

## Common errors

### `RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed).`

A loop that computes the loss once, before the loop starts:

In [16]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
loss = ((w * latitude + b - january) ** 2).mean()
for step in range(2):
    loss.backward()

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

The first `backward()` walked the graph and freed the values it had saved, such as the errors that
the square needs for its derivative, so the second walk had nothing to multiply. Computing the loss
inside the loop builds a new graph in every step, which is what a training loop does.
`retain_graph=True` on the first call keeps the saved values, for the rare case that needs two
walks of one graph.

In [17]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
for step in range(2):
    loss = ((w * latitude + b - january) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= 0.1 * w.grad
        b -= 0.1 * b.grad
    w.grad.zero_()
    b.grad.zero_()
    print(step, round(loss.item(), 3))

0 34.569
1 23.759


Each step now has a graph of its own, and the loss falls from 34.569 to 23.759.

### `RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.`

An update written as plain arithmetic:

In [18]:
w = torch.tensor(0.0, requires_grad=True)
loss = ((w * latitude - january) ** 2).mean()
loss.backward()
w -= 0.1 * w.grad

RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.

`w -= ...` changes `w` in place while PyTorch is recording. The change would be recorded as an
operation, and `w` is a leaf, where gradients are collected, which cannot also be the result of one,
so PyTorch raises whether or not a graph uses `w`. The update is not part of the model, so it
belongs inside `torch.no_grad()`, where nothing is recorded.

In [19]:
with torch.no_grad():
    w -= 0.1 * w.grad
print(round(w.item(), 4))

-0.397


The step moved `w` from 0 to −0.397, a tenth of its gradient of 3.9696, in the downhill direction.

### `RuntimeError: A non-persistent GradientTape can only be used to compute one set of gradients (or jacobians)`

The two gradients asked for one at a time:

In [20]:
with tf.GradientTape() as tape:
    loss_tf = tf.reduce_mean((w_tf * latitude_tf + b_tf - january_tf) ** 2)
dw = tape.gradient(loss_tf, w_tf)
db = tape.gradient(loss_tf, b_tf)

RuntimeError: A non-persistent GradientTape can only be used to compute one set of gradients (or jacobians)

The first `tape.gradient` used up the tape. Ask for both gradients in one call, with a list, or make
the tape persistent when the gradients belong to different results.

In [21]:
with tf.GradientTape() as tape:
    loss_tf = tf.reduce_mean((w_tf * latitude_tf + b_tf - january_tf) ** 2)
dw, db = tape.gradient(loss_tf, [w_tf, b_tf])
print(round(float(dw), 4), round(float(db), 4))

3.9696 10.2153


One call with a list returns both gradients, the same 3.9696 and 10.2153 as before.

### No error, and a gradient three times too large

The same loss and its gradient, three times over, at the same slope of 0:

In [22]:
w = torch.tensor(0.0, requires_grad=True)
for step in range(3):
    loss = ((w * latitude - january) ** 2).mean()
    loss.backward()
    print(step, round(w.grad.item(), 4))

0 3.9696
1 7.9392
2 11.9088


Nothing in the loop changes `w`, so the loss is the same in all three steps, and the printed
gradient grows from 3.9696
to 7.9392 to 11.9088, because each `backward()` added its gradient to the one before. A loop that
stepped with the last of them would move three times too far, and nothing would say so.

In [23]:
w = torch.tensor(0.0, requires_grad=True)
for step in range(3):
    loss = ((w * latitude - january) ** 2).mean()
    loss.backward()
    print(step, round(w.grad.item(), 4))
    w.grad.zero_()

0 3.9696
1 3.9696
2 3.9696


Clearing `.grad` after each step leaves every gradient at 3.9696. An optimizer's `zero_grad()`,
which the **Losses and Optimizers** notebook uses, clears every weight's gradient at once.

## Recap

- You can read the record behind a PyTorch tensor from its `grad_fn`, and say which tensors are leaves.
- You can get gradients with `backward()` and `.grad`, or with a `tf.GradientTape`, and check them
  against a derivative worked out by hand.
- You can clear gradients before each step, and add them up on purpose when that is what you want.
- You can keep an update or an evaluation out of the record with `torch.no_grad()` or `detach()`.
- You can ask a tape for several gradients at once, or make it persistent.

## What is next

The **Gradient Descent by Hand** notebook builds a whole training loop from these pieces, with
nothing hidden, and shows what the learning rate and the features' scale do to it.

---

&#8592; **Previous:** [Shapes and Broadcasting](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/03-shapes-and-broadcasting.ipynb)  &nbsp;·&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Gradient Descent by Hand](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/05-gradient-descent-by-hand.ipynb) &#8594;
